# Stage 1 — SHADOW ONLY temporal CAM quick run

Single Shadow Hand run for the temporal human encoder. Uses `--human_encoder temporal_cam`
with `T=8` and 6000 steps, while keeping the Shadow-only metric/training setup
otherwise aligned with the spatial baseline notebook.


## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU found. Go to Runtime > Change runtime type and select T4 GPU.')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/AIST-hand')
GITHUB_TOKEN = ''
GITHUB_USER  = 'isapedraza'
REPO_NAME    = 'AIST-hand'

# ── Robot config ──────────────────────────────────────────────────────────────
ROBOTS = ['shadow']

# ── Human data ────────────────────────────────────────────────────────────────
HUMAN_ROT_REPR    = 'r6'
CSV_PATH          = DRIVE_ROOT / 'hograspnet_abl14_r6.csv'
EXTRA_HUMAN_CSV   = DRIVE_ROOT / 'data/processed/hagrid_dong_r6.csv'
EXTRA_HUMAN_RATIO = 0.10

# ── Human encoder ─────────────────────────────────────────────────────────────
HUMAN_ENCODER   = 'temporal_cam'
TEMPORAL_WINDOW = 8

# ── Checkpoint ────────────────────────────────────────────────────────────────
CKPT_PATH   = DRIVE_ROOT / 'checkpoints/stage1_shadow_temporal_cam_6000.pt'
RESUME_CKPT = None

# ── Training hyperparams ──────────────────────────────────────────────────────
B              = 50_000
N_STEPS        = 6_000
LOG_EVERY      = 50
CKPT_EVERY     = 500
VAL_EVERY      = 0
N_EVAL_BATCHES = 20
B_EVAL         = 5000
SEED           = -1

Z_DIM      = 16
SHARED_DIM = 1024
LR         = 1e-3
LAMBDA_C   = 10.0
LAMBDA_REC = 5.0
LAMBDA_LTC = 1.0
LAMBDA_TMP = 0.1
W_R        = 1.0
W_JOINTS   = 1.2
W_AHG      = 0.07
N_TRIPLETS = None
MARGIN     = 0.1

# -- Cross-robot S_k metric (Xin tip/pinch + adaptive D_R) ---------------------
# Calibrated via least-squares regression on 10k shadow pose pairs to approximate
# Run20 S_k (W_R=1.0*D_R + W_JOINTS=1.2*D_joints). Key finding: d_tip_rot feature
# magnitude ~1.2 vs D_R ~0.08 -> with lam_tip_rot=0.3 tip_rot dominated 4x D_R,
# adding noise (regression optimal = -0.025 -> zero). lam_pinch 1.3 -> 0.07.
# lam_dr 1.2 -> 1.4. R² with these lambdas = 0.74 vs Run20 (structural ceiling).
# Previous config R² = -28 (tip_rot dominated everything).
SK_METRIC     = 'xin'
LAM_TIP       = 1.0
LAM_THUMB_TIP = 1.3
LAM_PINCH     = 0.07
LAM_TIP_ROT   = 0.0
LAM_DR        = 1.4
LAM_FINGER    = 0.0
MEM_DEBUG     = True

print(f'Drive root : {DRIVE_ROOT}')
print(f'Robots     : {ROBOTS}')
print(f'B={B}  N_STEPS={N_STEPS}  VAL_EVERY={VAL_EVERY}')
print(f'HUMAN_ENCODER={HUMAN_ENCODER}  TEMPORAL_WINDOW={TEMPORAL_WINDOW}')
print(f'LAM_TIP={LAM_TIP}  LAM_THUMB_TIP={LAM_THUMB_TIP}  LAM_PINCH={LAM_PINCH}  LAM_TIP_ROT={LAM_TIP_ROT}  LAM_DR={LAM_DR}')

## 3. Clone repo from GitHub

In [ ]:
REPO_DIR = f'/content/{REPO_NAME}'
BRANCH   = 'main'

if not os.path.exists(REPO_DIR):
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} fetch origin')

os.system(f'git -C {REPO_DIR} checkout {BRANCH}')
os.system(f'git -C {REPO_DIR} pull origin {BRANCH}')
print(f'Checked out branch: {BRANCH}')

REPO_ROOT = Path(REPO_DIR)
print(f'Repo: {REPO_ROOT}')

## 4. Install dependencies

In [ ]:
import torch
print(f'torch={torch.__version__}  cuda={torch.version.cuda}')

# setuptools-scm required by latent-retargeting pyproject.toml
!pip install -q setuptools-scm
!pip install -q pytorch-kinematics
!pip install -q mujoco   # EIGENGRASP_ONLINE collision filter
!pip install -q -e /content/AIST-hand/models/latent-retargeting
!pip install -q -e /content/AIST-hand/human

print('Done.')

## 5. Validate Drive files + write robots YAML

In [ ]:
import psutil, shutil, yaml

ram = psutil.virtual_memory()
print(f'RAM total : {ram.total / 1e9:.1f} GB')
print(f'RAM avail : {ram.available / 1e9:.1f} GB')
print(f'RAM used  : {ram.used / 1e9:.1f} GB')

# Stage gitignored valid_poses NPZ from Drive into the repo paths the yaml expects.
# Upload the three *_1M_dong.npz to DRIVE_ROOT (root) before running.
VALID_POSES_DRIVE = DRIVE_ROOT
for name in ROBOTS:
    ycfg = yaml.safe_load(open(REPO_ROOT / f'robot/hand-configs/{name}.yaml'))
    vp = ycfg.get('valid_poses')
    if not vp:
        raise ValueError(f'{name}.yaml has no valid_poses -> would fall into EIGENGRASP_ONLINE (slow).')
    dst = REPO_ROOT / vp
    if dst.exists():
        print(f'{name}: valid_poses present ({dst.name})')
        continue
    src = VALID_POSES_DRIVE / dst.name
    if not src.exists():
        raise FileNotFoundError(f'{name}: upload {src} to Drive (gitignored, not in clone)')
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dst)
    print(f'{name}: staged {src.name} -> {dst}')

required = [CSV_PATH, EXTRA_HUMAN_CSV]
for name in ROBOTS:
    required.append(REPO_ROOT / f'robot/hand-configs/{name}.yaml')
missing = [str(p) for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError('Missing files:\n' + '\n'.join(missing))
print('All required files found.')


## 6. Run Stage 1 training

In [ ]:
import subprocess, sys

print(f'HUMAN_ROT_REPR : {HUMAN_ROT_REPR}')
print(f'HUMAN_ENCODER  : {HUMAN_ENCODER}')
print(f'TEMPORAL_WINDOW: {TEMPORAL_WINDOW}')
print(f'CSV            : {CSV_PATH}')
print(f'CKPT           : {CKPT_PATH}')
print(f'RESUME_CKPT    : {RESUME_CKPT}')
print(f'Robots         : {ROBOTS}')
print(f'Z_DIM={Z_DIM}  SHARED_DIM={SHARED_DIM}  MARGIN={MARGIN}')
print(f'SEED           : {SEED} ({"random" if SEED < 0 else "fixed"})')

cmd = [
    sys.executable, '-u',
    str(REPO_ROOT / 'models/latent-retargeting/scripts/train_cross_emb.py'),
    '--repo_root',         str(REPO_ROOT),
    '--csv_path',          str(CSV_PATH),
    '--ckpt_path',         str(CKPT_PATH),
    '--robots',            *ROBOTS,
    '--extra_human_csv',   str(EXTRA_HUMAN_CSV),
    '--extra_human_ratio', str(EXTRA_HUMAN_RATIO),
    '--human_rot_repr',    HUMAN_ROT_REPR,
    '--human_encoder',      HUMAN_ENCODER,
    '--temporal_window',    str(TEMPORAL_WINDOW),
    '--b',                 str(B),
    '--n_steps',           str(N_STEPS),
    '--log_every',         str(LOG_EVERY),
    '--ckpt_every',        str(CKPT_EVERY),
    '--z_dim',             str(Z_DIM),
    '--shared_dim',        str(SHARED_DIM),
    '--lr',                str(LR),
    '--lambda_c',          str(LAMBDA_C),
    '--lambda_rec',        str(LAMBDA_REC),
    '--lambda_ltc',        str(LAMBDA_LTC),
    '--lambda_tmp',        str(LAMBDA_TMP),
    '--sk_metric',         SK_METRIC,
    '--lam_tip',           str(LAM_TIP),
    '--lam_thumb_tip',     str(LAM_THUMB_TIP),
    '--lam_pinch',         str(LAM_PINCH),
    '--lam_tip_rot',       str(LAM_TIP_ROT),
    '--lam_dr',            str(LAM_DR),
    '--lam_finger',        str(LAM_FINGER),
    '--margin',            str(MARGIN),
    '--seed',              str(SEED),
    '--val_every',         str(VAL_EVERY),
    '--n_eval_batches',    str(N_EVAL_BATCHES),
    '--b_eval',            str(B_EVAL),
    '--log_metric_stats',
    '--skip_final_eval',
    '--force_cross_robot',   # same metric as the 3-hand run (mini-tips + adaptive D_R)
] + (['--mem_debug'] if MEM_DEBUG else [])
if N_TRIPLETS is not None:
    cmd.extend(['--n_triplets', str(N_TRIPLETS)])
if RESUME_CKPT is not None:
    cmd.extend(['--resume_ckpt', str(RESUME_CKPT)])

print('\nTraining command:')
print(' '.join(cmd))

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Training failed with code {proc.returncode}')

## 7. Memory diagnostics

In [ ]:
import torch, psutil

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total_gpu = torch.cuda.get_device_properties(0).total_memory / 1e9
ram       = psutil.virtual_memory()

print(f'GPU allocated : {allocated:.2f} GB')
print(f'GPU reserved  : {reserved:.2f} GB')
print(f'GPU total     : {total_gpu:.2f} GB')
print(f'GPU free      : {total_gpu - reserved:.2f} GB')
print(f'RAM used      : {ram.used / 1e9:.1f} GB / {ram.total / 1e9:.1f} GB')